# Notebook 8.1  A denoising front end and an honest evaluation on noisy Arabic speech

**Goal.** Build a single-channel denoiser, apply it to noise-augmented (and reverberant) Arabic audio, and measure both signal quality (SI-SDR, and PESQ/STOI when available) and the downstream word error rate, then plot error against signal-to-noise ratio with and without enhancement, as in Figure 8.5.

**How to use real data.** The cells below run end to end on a small *synthetic* clip so the notebook works with no downloads. Each data step says how to swap in real audio: clean speech from [Common Voice Arabic](https://commonvoice.mozilla.org/ar) and noise from [MUSAN](https://www.openslr.org/17/), subject to their current licenses. Replace `make_synthetic_speech()` with a real `.wav` read and the rest of the pipeline is unchanged.

This notebook accompanies Chapter 8. It is deliberately small and CPU-only; the point is the *method*, not state-of-the-art numbers.

## 1. Setup

Only NumPy, SciPy, and Matplotlib are required. PESQ and STOI are optional; the notebook detects them and skips gracefully if they are missing.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # works in any environment; remove for interactive plots
import matplotlib.pyplot as plt
from scipy import signal

rng = np.random.default_rng(0)
SR = 16000  # 16 kHz, the standard for ASR (Chapter 3)

# Optional perceptual metrics. Install with: pip install pesq pystoi
try:
    from pesq import pesq as _pesq
    HAVE_PESQ = True
except Exception:
    HAVE_PESQ = False
try:
    from pystoi import stoi as _stoi
    HAVE_STOI = True
except Exception:
    HAVE_STOI = False
print('PESQ available:', HAVE_PESQ, '| STOI available:', HAVE_STOI)

## 2. A clean signal to work with

For a reproducible demo we synthesize a voiced, speech-like signal: a moving fundamental frequency with a few harmonics, shaped by a slow amplitude envelope so it has speech-like pauses. 

**To use real Arabic speech instead**, replace the body of `make_synthetic_speech` with:
```python
import soundfile as sf
wav, sr = sf.read('clip.wav')
if sr != SR:
    wav = signal.resample_poly(wav, SR, sr)
return wav.astype(np.float32)
```

In [ ]:
def make_synthetic_speech(seconds=3.0, sr=SR):
    t = np.arange(int(seconds*sr)) / sr
    f0 = 130 + 25*np.sin(2*np.pi*0.7*t)            # wandering pitch
    voiced = sum((1.0/h)*np.sin(2*np.pi*h*f0*t) for h in range(1,6))
    env = (0.5 + 0.5*np.sin(2*np.pi*1.3*t)).clip(0, 1) ** 2  # syllable-like envelope
    sig = env * voiced
    sig = sig / (np.max(np.abs(sig)) + 1e-9)
    return sig.astype(np.float32)

clean = make_synthetic_speech()
print('clean:', clean.shape, 'duration %.2fs' % (len(clean)/SR))

## 3. Degrade the audio: additive noise at a target SNR, plus reverberation

`add_noise_at_snr` scales the noise so the mixture has exactly the requested signal-to-noise ratio (SNR). `apply_reverb` convolves the signal with a synthetic room impulse response (an exponentially decaying noise burst); swap in a measured RIR file to model a specific room. Use **separate** noise clips and RIRs for training, development, and test so the evaluation does not leak acoustic conditions (Chapter 8, Dataset Spotlight).

In [ ]:
def add_noise_at_snr(clean, noise, snr_db):
    # tile/trim noise to match length
    if len(noise) < len(clean):
        noise = np.tile(noise, int(np.ceil(len(clean)/len(noise))))
    noise = noise[:len(clean)]
    p_clean = np.mean(clean**2) + 1e-12
    p_noise = np.mean(noise**2) + 1e-12
    target = p_clean / (10**(snr_db/10))      # required noise power
    noise = noise * np.sqrt(target / p_noise)
    return (clean + noise).astype(np.float32)

def make_rir(sr=SR, rt60=0.3):
    n = int(rt60*sr)
    decay = np.exp(-np.arange(n) * (6.9 / (rt60*sr)))   # ~60 dB decay over rt60
    rir = rng.standard_normal(n) * decay
    rir[0] = 1.0                                        # direct path
    return (rir / np.sqrt(np.sum(rir**2))).astype(np.float32)

def apply_reverb(x, rir):
    y = signal.fftconvolve(x, rir)[:len(x)]
    return y.astype(np.float32)

# white-ish noise stands in for MUSAN; replace with a real noise clip read
noise = rng.standard_normal(len(clean)).astype(np.float32)
noisy = add_noise_at_snr(clean, noise, snr_db=5.0)
reverb = apply_reverb(clean, make_rir())
print('noisy and reverberant signals ready')

## 4. SI-SDR as a five-step algorithm

This is the metric described in Section 8.6, written exactly as the five steps: (1) find the scaling factor from the dot products, (2) scale the target, (3) isolate the residual noise, (4) form the energy ratio, (5) convert to decibels. The function is the one Exercise 2 asks you to build and test.

In [ ]:
def si_sdr(estimate, target, eps=1e-9):
    estimate = np.asarray(estimate, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    # 1. scaling factor: <est, tgt> / <tgt, tgt>
    alpha = np.dot(estimate, target) / (np.dot(target, target) + eps)
    # 2. scaled target
    scaled_target = alpha * target
    # 3. residual noise
    noise = estimate - scaled_target
    # 4. energy ratio
    ratio = (np.sum(scaled_target**2) + eps) / (np.sum(noise**2) + eps)
    # 5. decibels
    return 10.0 * np.log10(ratio)

print('SI-SDR(noisy, clean)  = %6.2f dB' % si_sdr(noisy, clean))

### 4a. Exercise 2 check: scale invariance

Multiplying the estimate by a constant (here 5.0) or inverting its phase (multiply by -1) must leave SI-SDR unchanged, because the scaling factor is projected out in step 1.

In [ ]:
base = si_sdr(noisy, clean)
scaled = si_sdr(5.0*noisy, clean)
flipped = si_sdr(-1.0*noisy, clean)
print('original : %.6f dB' % base)
print('x 5.0    : %.6f dB' % scaled)
print('x -1.0   : %.6f dB' % flipped)
assert np.allclose([base, base], [scaled, flipped], atol=1e-6), 'SI-SDR should be scale/phase invariant'
print('PASS: SI-SDR is invariant to scale and phase')

## 5. A simple denoiser: spectral subtraction

No training is needed. We take the short-time Fourier transform (STFT), estimate the noise magnitude from the quietest frames, subtract it from every frame, floor the result so it stays non-negative, and invert the STFT. This is the classic baseline of Section 8.2; a learned mask network would replace the subtraction step but keep the same analyze, mask, synthesize shape (Figure 8.1).

In [ ]:
def spectral_subtraction(noisy, sr=SR, n_fft=512, hop=128, over_sub=1.5, floor=0.05):
    f, t, Z = signal.stft(noisy, fs=sr, nperseg=n_fft, noverlap=n_fft-hop)
    mag, phase = np.abs(Z), np.angle(Z)
    # estimate noise from the 10% lowest-energy frames
    frame_energy = mag.sum(axis=0)
    quiet = mag[:, frame_energy <= np.quantile(frame_energy, 0.10)]
    noise_mag = quiet.mean(axis=1, keepdims=True) if quiet.size else mag.mean(axis=1, keepdims=True)
    clean_mag = np.maximum(mag - over_sub*noise_mag, floor*mag)
    _, y = signal.istft(clean_mag*np.exp(1j*phase), fs=sr, nperseg=n_fft, noverlap=n_fft-hop)
    return y[:len(noisy)].astype(np.float32)

enhanced = spectral_subtraction(noisy)
print('SI-SDR before enhancement: %6.2f dB' % si_sdr(noisy, clean))
print('SI-SDR after  enhancement: %6.2f dB' % si_sdr(enhanced, clean))

## 6. Quality and intelligibility metrics (optional)

If PESQ and STOI are installed, we report them against the clean reference. They need a clean reference, which simulated mixtures provide but real field recordings do not (Section 8.6).

In [ ]:
def report_metrics(name, est, ref, sr=SR):
    row = {'signal': name, 'SI-SDR (dB)': round(si_sdr(est, ref), 2)}
    if HAVE_PESQ:
        try:
            row['PESQ'] = round(_pesq(sr, ref.astype(np.float64), est.astype(np.float64), 'wb'), 2)
        except Exception as e:
            row['PESQ'] = 'n/a'
    if HAVE_STOI:
        try:
            row['STOI'] = round(_stoi(ref, est, sr, extended=False), 2)
        except Exception:
            row['STOI'] = 'n/a'
    return row

for r in [report_metrics('noisy', noisy, clean), report_metrics('enhanced', enhanced, clean)]:
    print(r)

## 7. Downstream error vs. SNR, with and without enhancement

The decisive metric for an ASR front end is the recognizer error rate (Section 8.6), so we sweep several SNRs and measure error with and without the denoiser.

**Real recognizer.** If `faster-whisper` or `transformers` is installed and you pass real Arabic clips, set `USE_REAL_ASR = True` and fill in `transcribe()`; then `error_rate` becomes a true word error rate (use the normalization of Chapter 4). 

**Default (offline) mode.** With the synthetic clip there is no transcript, so we use a transparent *proxy* for recognizer error: a monotone function of SI-SDR, clearly labelled illustrative. It reproduces the shape of Figure 8.5 (gains largest at low SNR) without pretending to be a measured WER.

In [ ]:
USE_REAL_ASR = False  # set True only with a real recognizer and real transcripts

def transcribe(wav, sr=SR):
    raise NotImplementedError('Plug in faster-whisper/transformers here for real WER.')

def proxy_error_from_sisdr(sdr_db):
    # illustrative only: maps signal quality to an error rate in [~5%, ~80%]
    return float(np.clip(80.0 / (1.0 + np.exp((sdr_db + 2.0)/3.0)) + 5.0, 0, 100))

snrs = [-5, 0, 5, 10, 15, 20]
err_noisy, err_enh = [], []
for snr in snrs:
    nz = add_noise_at_snr(clean, rng.standard_normal(len(clean)).astype(np.float32), snr)
    en = spectral_subtraction(nz)
    if USE_REAL_ASR:
        # err = wer(reference_text, transcribe(nz)); etc.
        raise SystemExit('Provide reference transcripts and a recognizer to use real WER.')
    err_noisy.append(proxy_error_from_sisdr(si_sdr(nz, clean)))
    err_enh.append(proxy_error_from_sisdr(si_sdr(en, clean)))

for s, a, b in zip(snrs, err_noisy, err_enh):
    print('SNR %3d dB | no enh %5.1f%% | with enh %5.1f%%' % (s, a, b))

In [ ]:
fig, ax = plt.subplots(figsize=(7,4.2))
ax.plot(snrs, err_noisy, 'o-', label='no enhancement')
ax.plot(snrs, err_enh, 's--', label='with enhancement')
ax.set_xlabel('signal-to-noise ratio (dB), low to high')
ax.set_ylabel('recognizer error (%), lower is better')
ax.set_title('Error vs. SNR (illustrative proxy; see Section 7)')
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig('error_vs_snr.png', dpi=120)
print('saved error_vs_snr.png; gap is largest at low SNR, as in Figure 8.5')

## 8. Where to go next

- Replace the synthetic clip and noise with real Common Voice Arabic and MUSAN, and report per-dialect numbers.
- Swap spectral subtraction for a learned mask network (Figure 8.1) and compare SI-SDR and WER.
- Turn on a real recognizer to replace the proxy error with a measured word error rate, using the Chapter 4 normalization.
- For two-speaker mixtures, add a separation model and score it with permutation-invariant SI-SDR (Exercise 3).